# Notebook 078: Bucketing for Pre-Sorted Join Optimization and Shuffle Elimination

**Module:** Module 08: Performance Tuning, Query Optimization & Skew Mitigation  
**Difficulty:** `Advanced`  
**Databricks Masterclass Reference:** Video 13: Databricks Clusters & Video 23: Databricks Interview Questions  

---

## 1. Business Scenario & Real-World Context
Two multi-billion-row fact tables (Orders and Shipments) are joined hourly in a critical financial pipeline. Because both tables are huge, broadcast joins are impossible. Every run shuffles 5TB of data across the network, consuming 40 minutes of compute.

## 2. Technical Requirement & Problem Statement
Eliminate shuffle using Spark Bucketing: create bucketed tables with `bucketBy()` and `sortBy()`, configure matching bucket counts on both tables, and verify the total elimination of the Exchange (shuffle) operator from Sort-Merge Joins.

## 3. Spark & Databricks Architecture Notes
Bucketing pre-shuffles and pre-sorts data on write according to a hash function of the bucket column. When two tables bucketed on the same key with identical (or multiple) bucket counts are joined, Spark reads matching bucket files directly without triggering any network shuffle (`Exchange` operator disappears).


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_078_Bucketing_for_Pre-Sorted_Join_Optimization_and_Shuffle_Elimination")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Creating Bucketed Orders and Shipments Tables
Write tables bucketed into 4 buckets on join key 'order_id'.

In [ ]:
# Ensure Hive support or managed catalog enables bucketing
spark.sql("CREATE DATABASE IF NOT EXISTS bucket_db_078")
spark.sql("USE bucket_db_078")

# Create bucketed orders table
orders = [(f"ORD-{i:03d}", 100.0 + i) for i in range(1, 21)]
df_ord = spark.createDataFrame(orders, ["order_id", "amount"])

df_ord.write.format("parquet")\
    .bucketBy(4, "order_id")\
    .sortBy("order_id")\
    .mode("overwrite")\
    .saveAsTable("bucket_db_078.t_orders")

# Create bucketed shipments table with identical bucket count (4)
shipments = [(f"ORD-{i:03d}", f"CARRIER-{i%3}") for i in range(1, 21)]
df_ship = spark.createDataFrame(shipments, ["order_id", "carrier"])

df_ship.write.format("parquet")\
    .bucketBy(4, "order_id")\
    .sortBy("order_id")\
    .mode("overwrite")\
    .saveAsTable("bucket_db_078.t_shipments")

print("Both tables bucketed into 4 buckets and sorted on order_id.")

## 6. Verifying Shuffle Elimination in Physical Join Plan
Join bucketed tables and inspect physical plan.

In [ ]:
# Enable bucketing optimization
spark.conf.set("spark.sql.sources.bucketing.enabled", "true")

df_b1 = spark.table("bucket_db_078.t_orders")
df_b2 = spark.table("bucket_db_078.t_shipments")

df_joined_bucket = df_b1.join(df_b2, on="order_id", how="inner")

print("=== PHYSICAL PLAN (Notice absence of 'Exchange hashpartitioning' shuffle!) ===")
df_joined_bucket.explain(mode="simple")
df_joined_bucket.show(5)

## 7. Clean Up Database
Drop temporary bucketed tables.

In [ ]:
spark.sql("DROP TABLE IF EXISTS bucket_db_078.t_orders")
spark.sql("DROP TABLE IF EXISTS bucket_db_078.t_shipments")
print("Cleaned up temporary tables.")

## 9. Key Architectural Takeaways
- **Bucketing pre-sorts and pre-distributes data on write, eliminating network shuffle during downstream joins.**
- **Both tables must be bucketed on the exact same join key and have matching or multiple bucket counts.**
- **Bucketing requires managed tables (`saveAsTable`); it does not apply to unmanaged path-based writes.**

## 10. Senior Data Engineering Interview Questions
1. How does bucketing eliminate the `Exchange` operator in a Sort-Merge Join physical plan?
2. What happens if two bucketed tables are joined, but Table A has 8 buckets and Table B has 16 buckets?
3. Why is bucketing less common in modern Databricks compared to Z-Ordering and Liquid Clustering?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
